# Разговорная рекомендация музыки: эксперименты

Датасет **Music4All-CRS** (`docs/dataset.md`): у пользователя есть история прослушиваний Last.fm и текстовый профиль,
на каждый трек, который он послушал в целевом месяце, есть синтетический **запрос на русском**.
Задача: по запросу + истории + профилю найти этот трек в каталоге из 64k треков.

Особенности, от которых зависит решение:
* одна цель на запрос; 11 типов запросов (`query_family`): точное название, строчка текста, жанр, настроение, эпоха...
* **93% целей уже есть в истории** пользователя (`is_new = False`), поэтому прослушанное не выкидываем;
* запросы на русском, а теги каталога на английском.

![scheme](scheme.png)

| Шаг | Модуль | Сейчас |
|---|---|---|
| 1. разбор запроса (llm-саммари) | `recsys/dialog.py` + словарь `recsys/ru.py` | правила для русского, LLM по флагу |
| 2. кандидаты | `recsys/retrieval/sources.py` | bm25, relisten (своя история), title (название, транслит), audio (эмбеддинги MuQ), history, popular; lyrics по флагу |
| 3. RRF + фильтры + дедуп | `recsys/fusion.py` | настоящий |
| 4. ранкер | `recsys/ranking.py` | **заглушка** (порядок RRF), `heuristic`, клиент `api` |
| 5. описание от LLM | `recsys/explain.py` | **заглушка** (приветствие + список), LLM по флагу |

Код живёт в `.py`, ноутбук только импортирует и запускает. Здесь — разбор шагов, метрики и сравнение вариантов;
запуск на одном запросе — `inference.ipynb`, проверка сервисов BM25 и HNSW руками — `examples/candgen.ipynb`. Кандгены BM25 и HNSW работают на VPS двумя отдельными сервисами (`deploy/README.md`) и вызываются по HTTP, если включить `USE_CANDGEN`.

Установка зависимостей (быстро, если всё уже стоит).

In [ ]:
%pip install -q -r requirements.txt

Импорты. `autoreload` подхватывает правки в `.py` без перезапуска ядра.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from recsys.config import load_config
from recsys.data import load_data, history_frame
from recsys.pipeline import Pipeline
from recsys.schemas import Request
from recsys.eval import evaluate, compare_configs, metrics_by

pd.set_option("display.max_colwidth", 80)

## Конфиг

Все настройки лежат в `configs/default.yaml`; нужное можно переопределить в `OVERRIDES`.

* `data.source`: `synthetic` — сгенерированные данные в формате датасета (работает сразу), `crs` — настоящие файлы из `data.crs.dir`
* `data.crs.n_users`: сколько пользователей брать из каждого сплита (полный train ~12.9k), `data.max_positives_per_user`: сколько запросов на пользователя
* `retrieval.<источник>.enabled`, `ranker.type`, `summarizer.type` / `explainer.type` (`llm` — модель в DataSphere)
* `USE_CANDGEN = True`: BM25 и HNSW берутся по HTTP с VPS, адреса — переменные окружения `BM25_URL` (`http://<IP>:8001`) и `HNSW_URL` (`http://<IP>:8002`), API — `docs/candgen_api.md`; эмбеддинги тогда в память не грузятся. Без флага работают локальные `bm25` и `audio`

In [ ]:
OVERRIDES = {
    # настоящие данные: tracks_meta.parquet, train.parquet, test_public.parquet в папке data/
    # "data": {"source": "crs", "crs": {"dir": "data", "n_users": 300}},
    # поиск по строчке текста песни:
    # "data": {"crs": {"with_lyrics": True}}, "retrieval": {"lyrics": {"enabled": True}},
    # "ranker": {"type": "heuristic"},
}
USE_CANDGEN = False  # True: BM25 и HNSW с VPS по HTTP (configs/candgen.yaml)

config_files = ["configs/default.yaml"] + (["configs/candgen.yaml"] if USE_CANDGEN else [])
cfg = load_config(config_files, OVERRIDES)
cfg["data"]["source"], {s: cfg["candgen"][s]["url"] for s in ("bm25", "hnsw")} if USE_CANDGEN else "кандгены локально"

## Данные

Каталог треков и запросы по сплитам: один запрос = один позитив пользователя (`data.requests` — сплит для оценки, `data.eval_split`).

In [ ]:
data = load_data(cfg)
catalog = data.catalog
display(catalog.stats().to_frame("value").T)
catalog.df[["track_id", "artist", "title", "tags", "genres", "release_year", "lang", "popularity"]].head()

### Как выглядит запрос

Запрос пользователя, его тип, профиль, история (уникальные треки: сколько раз и когда последний раз) и цель — трек, который нужно найти.

In [ ]:
r = data.requests[0]
print("запрос:", r.dialog[0].text)
print("тип:", r.meta["query_family"], "| новый трек:", r.meta["is_new"], "| цель:", r.meta["artist"], "-", r.meta["title"])
print("профиль:", r.user_info[:300], "...")
print("демография:", r.user_attrs)
history_frame(r).head()

## Пайплайн

In [ ]:
pipe = Pipeline.from_config(cfg, catalog)
print("источники:", [r.name for r in pipe.retrievers])
print("шаги:", type(pipe.summarizer).__name__, type(pipe.ranker).__name__, type(pipe.explainer).__name__)
for x in pipe.retrievers:  # кандгены на VPS: доступность и версия индекса
    if getattr(x, "remote", False):
        print(f"  {x.name}:", x.health())

### Один запрос из датасета

Саммари (что понял шаг 1), текст ответа и таблица треков. `target_rank` — место цели в выдаче (None — не попала в top-k).

In [ ]:
resp = pipe.run(r, debug=True)
target = r.target_ids[0]
print("SUMMARY:", resp.summary.text)
print("target_rank:", resp.track_ids.index(target) + 1 if target in resp.track_ids else None)
print()
print(resp.text)
resp.to_frame()

### Что происходит на каждом шаге

Сколько кандидатов дал каждый источник, на каком месте в нём оказалась цель, веса RRF, время шагов и признаки, которые видит ранкер.

In [ ]:
dbg = resp.debug
print("1. саммари:", resp.summary)
print("2. кандидатов по источникам:", dbg["n_candidates"])
print("   место цели в источниках:", {s: next((c.rank for c in cs if c.track_id == target), None)
                                    for s, cs in dbg["candidates"].items()})
print("3. веса RRF:", dbg["weights"], "| после фильтров:", dbg["n_fused"])
print("кандгены на VPS:", {x.name: x.last_error or "ok" for x in pipe.retrievers if hasattr(x, "last_error")})
print("время, мс:", {k: round(v * 1000, 1) for k, v in dbg["timings"].items()})
dbg["features"].head(10)

### Настоящие запросы из датасета

Если в `data.crs.dir` лежат файлы Music4All-CRS (целиком или частями), берём по 2 настоящих запроса каждого типа и смотрим, что из них понял шаг 1.
Полный поиск на них имеет смысл только с полным `tracks_meta.parquet` (`data.source: crs`): в сэмпле каталога 5 треков,
и цели в него не входят.

In [ ]:
from recsys.data.crs import parquet_files, read_split, requests_from_split

try:
    train_files = parquet_files(cfg["data"]["crs"]["dir"], "train")
except FileNotFoundError:
    train_files = None
if train_files:
    real = requests_from_split(read_split(train_files, n_users=300))
    rows = []
    for fam, group in pd.Series(real).groupby([q.meta["query_family"] for q in real]):
        for q in group.head(2):
            sm = pipe.summarizer.summarize(q)
            rows.append({"тип": fam, "запрос": q.dialog[0].text, "цель": f"{q.meta['artist']} — {q.meta['title']}",
                         "теги": sm.include_tags, "исключить": sm.exclude_tags,
                         "страна/язык/год": sm.countries + sm.languages + sm.years})
    display(pd.DataFrame(rows))
else:
    print("train не найден в data.crs.dir: положите туда файлы датасета")

## Офлайн-оценка

`evaluate` прогоняет запросы и считает hit/recall/ndcg/mrr@10 (цель одна, поэтому hit = recall) и recall каждого источника
кандидатов. Смотрим по типам запросов (`query_family`) и отдельно по новым и знакомым трекам (`is_new`).

На синтетике метрики ничего не говорят о реальном качестве: запросы построены по тем же тегам, по которым ищем.

In [ ]:
per_request, mean = evaluate(pipe, data.requests[:400])
display(mean.round(3).to_frame("value").T)
display(metrics_by(per_request, "query_family").round(2))
metrics_by(per_request, "is_new").round(2)

Сравнение вариантов: одни и те же запросы, разные конфиги. Значение — hit@10 по типам запросов, строка ALL — в среднем.

In [ ]:
variants = {
    "stub ranker": {},
    "heuristic ranker": {"ranker": {"type": "heuristic"}},
    "без relisten": {"retrieval": {"relisten": {"enabled": False}}},
    "без title": {"retrieval": {"title": {"enabled": False}}},
    "без audio": {"retrieval": {"audio": {"enabled": False}, "hnsw_audio": {"enabled": False}}},
    "без запроса (только история)": {"retrieval": {k: {"enabled": False} for k in
                                     ["bm25", "title", "bm25_genres", "bm25_tags", "bm25_title"]}},
}
compare_configs(cfg, variants, catalog, data.requests[:400], by="query_family").round(2)

## LLM (саммари и описание)

Нужна GPU-конфигурация DataSphere и `transformers` (см. requirements.txt). Поставьте `USE_LLM = True`.
Модель загружается один раз и работает на шагах 1 и 5. Если LLM вернула не JSON, саммари берётся из правил.

In [ ]:
USE_LLM = False

if USE_LLM:
    from recsys.llm import build_llm
    cfg_llm = load_config(config_files, {
        **OVERRIDES,
        "llm": {"type": "local", "model_name": "Qwen/Qwen2.5-1.5B-Instruct"},
        "summarizer": {"type": "llm"},
        "explainer": {"type": "llm"},
    })
    llm = build_llm(cfg_llm)
    pipe_llm = Pipeline.from_config(cfg_llm, catalog, llm=llm, bm25_index=pipe.bm25_index)
    resp_llm = pipe_llm.run(r, debug=True)
    print("RAW LLM SUMMARY:", pipe_llm.summarizer.last_raw)
    print("SUMMARY:", resp_llm.summary)
    print(resp_llm.text)
    display(resp_llm.to_frame())

## Как заменить заглушку

* **Ранкер**: класс с `rank(features, ctx) -> DataFrame` (колонка `rank_score`) в `recsys/ranking.py`, признаки в `build_features` там же. Для внешнего сервиса: `ranker.type: api`, `ranker.api_url`.
* **Новый источник кандидатов**: класс-наследник `BaseRetriever` с `search(ctx) -> [Candidate]` в `recsys/retrieval/`, зарегистрировать в `build_retrievers` (`recsys/retrieval/sources.py`) и включить в `configs/default.yaml`.
* **Разбор запроса**: словарь `recsys/ru.py` или `summarizer.type: llm`.